# 02 - Baseline Evaluation (Colab)

Runs on **Qwen2.5-1.5B-Instruct** (no fine-tuning yet). Produces `eval/results/base/metrics.json` and `eval/results/base/timing.json`.

**Before running:** `Runtime > Change runtime type > T4 GPU` (or A100 if you have Colab Pro).

**Getting the repo into Colab** (pick one, run in the next cell):
- **Option A - GitHub (default):** the repo is public, so `REPO_URL` below is already filled in and just clones directly.
- **Option B - Drive upload:** zip `efficient-slm-benchmark/` and upload it to your Google Drive, then set `USE_DRIVE = True` and `DRIVE_ZIP_PATH` below. Saves results straight back to Drive so nothing is lost if the runtime disconnects.
- **Option C - none of the above:** leave both blank. The notebook falls back to hardcoded defaults (matching `configs/eval.yaml`/`configs/model.yaml`) and writes results under `/content/eval_results_base/` for you to download manually at the end.

In [ ]:
REPO_URL = "https://github.com/Shhaurya17/Efficient-Small-Language-Model-Adaptation-Quantization-Benchmark.git"  # public repo, no token needed
USE_DRIVE = False
DRIVE_ZIP_PATH = "/content/drive/MyDrive/efficient-slm-benchmark.zip"

import os

REPO_DIR = "/content/efficient-slm-benchmark"

if REPO_URL:
    !git clone -q {REPO_URL} {REPO_DIR}
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !unzip -q -o {DRIVE_ZIP_PATH} -d /content

HAVE_REPO = os.path.exists(os.path.join(REPO_DIR, "configs", "eval.yaml"))
print("Repo available:", HAVE_REPO)

In [ ]:
%%capture
!pip install -q transformers>=4.44.0 accelerate>=0.33.0 bitsandbytes>=0.43.0 lm-eval>=0.4.3 pyyaml

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
import sys
import yaml

DEFAULT_MODEL_CONFIG = {"model_name": "Qwen/Qwen2.5-1.5B-Instruct", "torch_dtype": "float16"}
DEFAULT_EVAL_CONFIG = {
    "benchmarks": [
        {"name": "mmlu", "num_fewshot": 5, "batch_size": 16},
        {"name": "arc", "num_fewshot": 5, "batch_size": 16},
        {"name": "gsm8k", "num_fewshot": 8, "batch_size": 4},
        {"name": "hellaswag", "num_fewshot": 10, "batch_size": 16},
    ]
}

if HAVE_REPO:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    with open(os.path.join(REPO_DIR, "configs", "model.yaml")) as f:
        model_config = yaml.safe_load(f)
    with open(os.path.join(REPO_DIR, "configs", "eval.yaml")) as f:
        eval_config = yaml.safe_load(f)
    RESULTS_DIR = os.path.join(REPO_DIR, "eval", "results", "base")
else:
    model_config = DEFAULT_MODEL_CONFIG
    eval_config = DEFAULT_EVAL_CONFIG
    RESULTS_DIR = "/content/eval_results_base"

os.makedirs(RESULTS_DIR, exist_ok=True)

try:
    from efficient_slm.inference.engine import load_model, measure_vram, measure_latency
except ImportError:
    def load_model(model_name, quantized=False, torch_dtype="float16", device_map="auto"):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        dtype = getattr(torch, torch_dtype)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map=device_map)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model.eval()
        return model, tokenizer

    def measure_vram(model):
        if not torch.cuda.is_available():
            return {"peak_vram_gb": None, "allocated_vram_gb": None}
        torch.cuda.synchronize()
        return {
            "peak_vram_gb": torch.cuda.max_memory_allocated(model.device) / 1e9,
            "allocated_vram_gb": torch.cuda.memory_allocated(model.device) / 1e9,
        }

    def measure_latency(model, tokenizer, batch_size=1, seq_length=256, num_samples=20):
        import time
        prompt = "The quick brown fox jumps over the lazy dog. " * 20
        inputs = tokenizer([prompt] * batch_size, return_tensors="pt", truncation=True, max_length=seq_length)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
        start = time.perf_counter()
        generated_tokens = 0
        with torch.no_grad():
            for _ in range(num_samples):
                out = model.generate(**inputs, max_new_tokens=32, do_sample=False, pad_token_id=tokenizer.eos_token_id)
                generated_tokens += out.shape[1] - inputs["input_ids"].shape[1]
        elapsed = time.perf_counter() - start
        return {
            "batch_size": batch_size,
            "ms_per_token": (elapsed / generated_tokens) * 1000,
            "throughput_tokens_per_sec": generated_tokens / elapsed,
        }

print("Model:", model_config["model_name"])
print("Results dir:", RESULTS_DIR)

## Load model and profile VRAM / latency

In [ ]:
import gc
import time

load_start = time.perf_counter()
model, tokenizer = load_model(model_config["model_name"], torch_dtype=model_config.get("torch_dtype", "float16"))
load_time_sec = time.perf_counter() - load_start

vram = measure_vram(model)
latency_bs1 = measure_latency(model, tokenizer, batch_size=1)
latency_bs16 = measure_latency(model, tokenizer, batch_size=16)

timing = {
    "model_name": model_config["model_name"],
    "load_time_sec": load_time_sec,
    "vram": vram,
    "latency_batch1": latency_bs1,
    "latency_batch16": latency_bs16,
}
timing

In [ ]:
# Free the model before lm-eval loads its own copy, to avoid doubling VRAM usage
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Run the evaluation suite

Set `LIMIT` to a small integer (e.g. `200`) for a fast smoke-test run before committing to the full benchmark, which can take a long while even on a T4.

In [ ]:
LIMIT = None  # e.g. 200 for a quick trial run

TASK_NAME_MAP = {"mmlu": "mmlu", "arc": "arc_challenge", "gsm8k": "gsm8k", "hellaswag": "hellaswag"}
# Primary metric per task (gsm8k has no ",none" key; hellaswag/arc use length-normalized acc)
METRIC_KEY_MAP = {
    "mmlu": "acc,none",
    "arc": "acc_norm,none",
    "gsm8k": "exact_match,flexible-extract",
    "hellaswag": "acc_norm,none",
}

import lm_eval
from lm_eval.models.huggingface import HFLM

lm = HFLM(
    pretrained=model_config["model_name"],
    dtype=model_config.get("torch_dtype", "float16"),
)

scores = {}
raw_results = {}
for bench in eval_config["benchmarks"]:
    task = TASK_NAME_MAP[bench["name"]]
    print(f"Running {task} ({bench['num_fewshot']}-shot)...")
    result = lm_eval.simple_evaluate(
        model=lm,
        tasks=[task],
        num_fewshot=bench["num_fewshot"],
        batch_size=bench["batch_size"],
        limit=LIMIT,
    )
    raw_results[bench["name"]] = result["results"][task]
    metric_key = METRIC_KEY_MAP[bench["name"]]
    scores[bench["name"]] = result["results"][task][metric_key]
    print(f"  {bench['name']}: {scores[bench['name']]:.4f}")

scores

## Save results

In [ ]:
import json

metrics = {
    "checkpoint": "base",
    "model_name": model_config["model_name"],
    "scores": scores,
    "raw_results": raw_results,
    "eval_limit": LIMIT,
}

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, default=str)

with open(os.path.join(RESULTS_DIR, "timing.json"), "w") as f:
    json.dump(timing, f, indent=2)

print("Saved to", RESULTS_DIR)
print(json.dumps(metrics, indent=2, default=str))

## Getting results back to your local repo

- **Drive option:** if you mounted Drive and cloned/unzipped the repo under it, the files above are already saved inside your Drive-synced copy — just make sure `REPO_DIR` pointed into `/content/drive/MyDrive/...`.
- **GitHub option:** commit and push from Colab:
  ```bash
  %cd {REPO_DIR}
  !git add eval/results/base
  !git commit -m "Add baseline eval results"
  !git push
  ```
  then `git pull` locally.
- **No repo option:** zip and download:
  ```python
  !zip -r /content/eval_results_base.zip {RESULTS_DIR}
  from google.colab import files
  files.download("/content/eval_results_base.zip")
  ```
  then unzip into `eval/results/base/` locally.